#first take the stage table which has start and end dates for each stages(booting, flowering, #grainfill) and then the climate data that has min, max, mean temp 
#merge them using the date range from first file to second file
#then we have all data to run the model

In [ ]:
import pandas as pd
import os
from glob import glob

In [ ]:
#import stage file and climate file

#file paths
CLIMATE_DIR = "/group/moniergrp/dbaral"
file_path = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data")

#get all the files
climate_files = sorted(glob(file_path + "/LOCA_csv/*.csv"))
stage_files = sorted(glob(file_path + "/LOCA_stage_table/*.csv"))
stress_files = sorted(glob(file_path + "/LOCA_stage_stress_tables/*.csv"))


final_dfs = []

#loop over all files

for i in range(len(climate_files)):
    climate_fp = climate_files[i]
    climate = pd.read_csv(climate_files[i], parse_dates = ["date"])
    stages = pd.read_csv(stage_files[i], parse_dates = ["start_date", "end_date"])
    stress = pd.read_csv(stress_files[i])
    
    #merge climate and stage
    merged = climate.merge(stages, on = ["county", "year"], how = "left")
    merged = merged[(merged["date"] >= merged["start_date"]) & (merged["date"] <= merged["end_date"])]
    
    #split stages
    
    # suffix dictionary for stages
    stage_suffix = {
        "booting": "bo",
        "flowering": "fl",
        "grain_fill": "gf"
    }
    
    # function to split stage-specific data and rename columns
    def split_and_rename(df, stage, suffix):
        out = df[df["stage"] == stage].copy()
        out = out.rename(columns={
            "tmmn": f"tmmn_{suffix}",
            "tmmx": f"tmmx_{suffix}",
            "tmean": f"tmean_{suffix}"
        })
        return out
    
    # function to aggregate annual mean
    def aggregate_annual_mean(df, suffix):
        return df.groupby(["county", "year"], as_index=False).agg({
            f"tmmn_{suffix}": "mean",
            f"tmmx_{suffix}": "mean",
            f"tmean_{suffix}": "mean"
        })
        
    booting_df = split_and_rename(merged, "booting", "bo")
    flowering_df = split_and_rename(merged, "flowering", "fl")
    grainfill_df = split_and_rename(merged, "grain_fill", "gf")               

    # seasonal data May 11 to Oct 6
    seasonal_df = climate[
        ((climate["date"].dt.month > 5) | ((climate["date"].dt.month == 5) & (climate["date"].dt.day >= 11))) &
        ((climate["date"].dt.month < 10) | ((climate["date"].dt.month == 10) & (climate["date"].dt.day <= 6)))
    ]
    
    booting_yr = aggregate_annual_mean(booting_df, "bo")
    flowering_yr = aggregate_annual_mean(flowering_df, "fl")
    grainfill_yr = aggregate_annual_mean(grainfill_df, "gf")

    seasonal_yr = seasonal_df.groupby(["county", "year"], as_index=False).agg({
        "tmmn": "mean",
        "tmmx": "mean",
        "tmean": "mean"
    })
    
    # stress yearly aggregation
    stress_yr = stress.groupby(["county", "year"], as_index=False).agg({
        "cdstress_bo": "max",
        "cdstress_fl": "max",
        "htstress_fl": "max"
    })
    
    # merge all together
    final_df = (
        booting_yr
        .merge(flowering_yr, on=["county", "year"], how="inner")
        .merge(grainfill_yr, on=["county", "year"], how="inner")
        .merge(seasonal_yr, on=["county", "year"], how="inner")
        .merge(stress_yr, on=["county", "year"], how="inner")
    )
    
    # save files
    base = os.path.basename(climate_fp)
    model_name = base.split("_rice")[0]
    output_name = f"{model_name}_Lasso_Model_Input_2030_2100.csv"
    save_path = os.path.join(CLIMATE_DIR, "run_project", "input_data", "LOCA_model_input", output_name)
    final_df.to_csv(save_path, index = False)
